In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# ---- Clean + Stable Environment Setup ----
!pip -q uninstall -y sentence-transformers
!pip -q install "jedi>=0.16"

# Upgrade build tools (prevents wheel build issues)
!pip -q install --upgrade pip setuptools wheel

# Ensure compatible pandas version for Colab
!pip -q install pandas==2.2.2

# Install HuggingFace + evaluation stack with safe pinned versions
!pip -q install \
    tokenizers==0.15.2 \
    transformers==4.39.3 \
    accelerate \
    datasets \
    bert-score \
    evaluate \
    sentencepiece \
    tqdm

# ---- Verify versions ----
import torch, pandas, transformers, tokenizers
print("torch:", torch.__version__)
print("pandas:", pandas.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
!pip -q install "jedi>=0.16"
!pip -q install -U pip setuptools wheel
!pip -q install "numpy==1.26.4" "pandas>=2.0"

# spaCy + scispaCy stack (no scipy)
!pip -q install "spacy==3.7.5" "thinc==8.2.5" "scispacy==0.5.4"

# SciSpaCy model
!pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz

In [ ]:
!pip install spacy
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_lg-0.5.1.tar.gz

In [ ]:
# =========================
# Imports / setup for UMLS concept-level evaluation
# =========================

!pip install "spacy==3.7.5" "thinc==8.2.5" "scispacy==0.5.4" -q
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz -q

import numpy as np
import pandas as pd
import spacy
import scispacy
from scispacy.linking import EntityLinker

nlp = spacy.load("en_core_sci_scibert")

if "scispacy_linker" not in nlp.pipe_names:
    nlp.add_pipe(
        "scispacy_linker",
        config={
            "linker_name": "umls",
            "resolve_abbreviations": True,
            "max_entities_per_mention": 1,
        },
    )

def extract_umls_concepts(text):
    doc = nlp(str(text))
    cuis = set()

    for ent in doc.ents:
        if hasattr(ent._, "kb_ents"):
            for cui, score in ent._.kb_ents:
                cuis.add(cui)

    return cuis

def umls_prf1(reference_text, prediction_text):
    ref_set = extract_umls_concepts(reference_text)
    pred_set = extract_umls_concepts(prediction_text)

    if len(ref_set) == 0 and len(pred_set) == 0:
        precision, recall, f1 = 1.0, 1.0, 1.0
    elif len(ref_set) == 0 or len(pred_set) == 0:
        precision, recall, f1 = 0.0, 0.0, 0.0
    else:
        common = ref_set & pred_set
        precision = len(common) / len(pred_set) if len(pred_set) > 0 else 0.0
        recall = len(common) / len(ref_set) if len(ref_set) > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "reference_concepts": ref_set,
        "prediction_concepts": pred_set,
        "common_concepts": ref_set & pred_set,
        "hallucinated_concepts": pred_set - ref_set,
        "omitted_concepts": ref_set - pred_set,
    }

print("UMLS / SciSpaCy setup completed.")
print("Pipeline:", nlp.pipe_names)

In [ ]:
def extract_umls_concepts(text):
    doc = nlp(str(text))
    cuis = set()

    for ent in doc.ents:
        if hasattr(ent._, "kb_ents"):
            for cui, score in ent._.kb_ents:
                cuis.add(cui)

    return cuis

def umls_prf1(reference_text, prediction_text):
    ref_set = extract_umls_concepts(reference_text)
    pred_set = extract_umls_concepts(prediction_text)

    if len(ref_set) == 0 and len(pred_set) == 0:
        precision, recall, f1 = 1.0, 1.0, 1.0
    elif len(ref_set) == 0 or len(pred_set) == 0:
        precision, recall, f1 = 0.0, 0.0, 0.0
    else:
        common = ref_set & pred_set
        precision = len(common) / len(pred_set) if len(pred_set) > 0 else 0.0
        recall = len(common) / len(ref_set) if len(ref_set) > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "reference_concepts": ref_set,
        "prediction_concepts": pred_set,
        "common_concepts": ref_set & pred_set,
        "hallucinated_concepts": pred_set - ref_set,
        "omitted_concepts": ref_set - pred_set,
    }

print("UMLS / SciSpaCy setup completed.")
print("Pipeline:", nlp.pipe_names)

In [ ]:
# =========================
# Imports / setup for BERTScore
# =========================

!pip install bert-score -q

from transformers import AutoTokenizer
from bert_score import score as bertscore

BERTSCORE_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
NUM_LAYERS = 12
LANG = "en"

bs_tokenizer = AutoTokenizer.from_pretrained(BERTSCORE_MODEL, use_fast=True)

def truncate_for_bertscore(texts, tokenizer=bs_tokenizer, max_len=512):
    truncated_texts = []

    for text in texts:
        ids = tokenizer.encode(
            str(text),
            add_special_tokens=True,
            truncation=True,
            max_length=max_len
        )
        truncated_texts.append(
            tokenizer.decode(ids, skip_special_tokens=True)
        )

    return truncated_texts

print("BERTScore setup completed.")

In [ ]:
!pip install transformers datasets rouge-score --quiet


In [ ]:
# =========================
# Imports / setup for BLEU & ROUGE
# =========================

!pip install evaluate rouge_score -q

import evaluate

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

print("ROUGE and BLEU metrics loaded.")

In [ ]:
from datasets import load_dataset

# CHANGE THIS PATH to your file
data_path = "/content/drive/MyDrive/biomedical_text_generation/data/unseen/sum_unseen.jsonl"

dataset_dict = load_dataset(
    "json",
    data_files={"test": data_path}
)

test_dataset = dataset_dict["test"]

print("Dataset loaded successfully")
print("Number of examples:", len(test_dataset))
print("\nExample record:")
print(test_dataset[0])


In [108]:
index = 3775

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 🔁 CHANGE MODEL HERE to compare models
# model_checkpoint = "GanjinZero/biobart-v2-base"
#model_checkpoint = "QizhiPei/biot5-base"
# model_checkpoint = "GanjinZero/biobart-base"
#model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final"
model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum_fix_final"


tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_checkpoint
).to("cuda")

model.eval()

print("Model loaded:", model_checkpoint)


In [110]:
def generate_summary(input_text,
                     max_input_length=1024,
                     max_new_tokens=256,
                     num_beams=4):

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to("cuda")

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
# 🔁 Change this number each time

input_text = test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT) BioBART v2:\n")
print(generated_text)

print("\n" + "="*100)


In [ ]:
# =========================
# Single-example evaluation
# =========================

# Assumes these already exist:
# target_text: reference summary
# generated_text: model output

reference = str(target_text)
prediction = str(generated_text)

# -------------------------
# ROUGE
# -------------------------
rouge_result = rouge_metric.compute(
    predictions=[prediction],
    references=[reference],
    use_stemmer=True
)

# -------------------------
# BLEU
# -------------------------
bleu_result = bleu_metric.compute(
    predictions=[prediction],
    references=[[reference]]
)

# -------------------------
# BERTScore
# -------------------------
P, R, F1 = bertscore(
    cands=[prediction],
    refs=[reference],
    model_type=BERTSCORE_MODEL,
    num_layers=NUM_LAYERS,
    lang="en",
    rescale_with_baseline=False,
    verbose=False
)

# -------------------------
# UMLS concept-level metrics
# -------------------------
target_concepts = extract_umls_concepts(reference)
generated_concepts = extract_umls_concepts(prediction)

target_set = set(target_concepts)
generated_set = set(generated_concepts)

common = target_set & generated_set

umls_precision = len(common) / len(generated_set) if len(generated_set) > 0 else 0.0
umls_recall = len(common) / len(target_set) if len(target_set) > 0 else 0.0
umls_f1 = (
    2 * umls_precision * umls_recall / (umls_precision + umls_recall)
    if (umls_precision + umls_recall) > 0
    else 0.0
)

hallucinated_concepts = generated_set - target_set
omitted_concepts = target_set - generated_set

# -------------------------
# Print results
# -------------------------
print("\n" + "=" * 80)
print("SINGLE EXAMPLE EVALUATION")
print("=" * 80)

print("\n--- ROUGE ---")
print(f"ROUGE-1: {rouge_result['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_result['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_result['rougeL']:.4f}")
print(f"ROUGE-Lsum: {rouge_result.get('rougeLsum', 0.0):.4f}")

print("\n--- BLEU ---")
print(f"BLEU: {bleu_result['bleu']:.4f}")

print("\n--- BERTScore ---")
print(f"Precision: {float(P.mean()):.4f}")
print(f"Recall:    {float(R.mean()):.4f}")
print(f"F1:        {float(F1.mean()):.4f}")

print("\n--- UMLS Concept-level Evaluation ---")
print(f"Precision: {umls_precision:.4f}")
print(f"Recall:    {umls_recall:.4f}")
print(f"F1-score:  {umls_f1:.4f}")

print("\n--- Concept Counts ---")
print(f"Target concepts:    {len(target_set)}")
print(f"Generated concepts: {len(generated_set)}")
print(f"Common concepts:    {len(common)}")
print(f"Hallucinated:       {len(hallucinated_concepts)}")
print(f"Omitted:            {len(omitted_concepts)}")

print("=" * 80)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 🔁 CHANGE MODEL HERE to compare models
# model_checkpoint = "GanjinZero/biobart-v2-base"
token_checkpoint = "QizhiPei/biot5-base"
#token_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_fix_final"
# model_checkpoint = "GanjinZero/biobart-base"
model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_fix_final"
#model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum_final"


tokenizer = AutoTokenizer.from_pretrained(token_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_checkpoint
).to("cuda")

model.eval()

print("Model loaded:", model_checkpoint)


In [114]:
def generate_summary(input_text,
                     max_input_length=512,
                     max_new_tokens=256,
                     num_beams=4):

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to("cuda")

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
# 🔁 Change this number each time

input_text = "summarize: " + test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT) BioT5:\n")
print(generated_text)

print("\n" + "="*100)


In [ ]:
# =========================
# Single-example evaluation
# =========================

# Assumes these already exist:
# target_text: reference summary
# generated_text: model output

reference = str(target_text)
prediction = str(generated_text)

# -------------------------
# ROUGE
# -------------------------
rouge_result = rouge_metric.compute(
    predictions=[prediction],
    references=[reference],
    use_stemmer=True
)

# -------------------------
# BLEU
# -------------------------
bleu_result = bleu_metric.compute(
    predictions=[prediction],
    references=[[reference]]
)

# -------------------------
# BERTScore
# -------------------------
P, R, F1 = bertscore(
    cands=[prediction],
    refs=[reference],
    model_type=BERTSCORE_MODEL,
    num_layers=NUM_LAYERS,
    lang="en",
    rescale_with_baseline=False,
    verbose=False
)

# -------------------------
# UMLS concept-level metrics
# -------------------------
target_concepts = extract_umls_concepts(reference)
generated_concepts = extract_umls_concepts(prediction)

target_set = set(target_concepts)
generated_set = set(generated_concepts)

common = target_set & generated_set

umls_precision = len(common) / len(generated_set) if len(generated_set) > 0 else 0.0
umls_recall = len(common) / len(target_set) if len(target_set) > 0 else 0.0
umls_f1 = (
    2 * umls_precision * umls_recall / (umls_precision + umls_recall)
    if (umls_precision + umls_recall) > 0
    else 0.0
)

hallucinated_concepts = generated_set - target_set
omitted_concepts = target_set - generated_set

# -------------------------
# Print results
# -------------------------
print("\n" + "=" * 80)
print("SINGLE EXAMPLE EVALUATION")
print("=" * 80)

print("\n--- ROUGE ---")
print(f"ROUGE-1: {rouge_result['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_result['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_result['rougeL']:.4f}")
print(f"ROUGE-Lsum: {rouge_result.get('rougeLsum', 0.0):.4f}")

print("\n--- BLEU ---")
print(f"BLEU: {bleu_result['bleu']:.4f}")

print("\n--- BERTScore ---")
print(f"Precision: {float(P.mean()):.4f}")
print(f"Recall:    {float(R.mean()):.4f}")
print(f"F1:        {float(F1.mean()):.4f}")

print("\n--- UMLS Concept-level Evaluation ---")
print(f"Precision: {umls_precision:.4f}")
print(f"Recall:    {umls_recall:.4f}")
print(f"F1-score:  {umls_f1:.4f}")

print("\n--- Concept Counts ---")
print(f"Target concepts:    {len(target_set)}")
print(f"Generated concepts: {len(generated_set)}")
print(f"Common concepts:    {len(common)}")
print(f"Hallucinated:       {len(hallucinated_concepts)}")
print(f"Omitted:            {len(omitted_concepts)}")

print("=" * 80)